In [2]:
from transformer import TransformerDecoder, generate_text_simple
import tiktoken
import torch
torch.manual_seed(42)

In [3]:
GPT2Config = {
  "activation_function": "gelu_new", # 'new'? prolly the tanh approximation
  "architectures": [
    "GPT2LMHeadModel" # anything special bout this?
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256, # same as eos?
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_layer": 12,
  "n_positions": 1024,
  "resid_pdrop": 0.1,
  "summary_activation": None,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": True,
  "summary_type": "cls_index",
  "summary_use_proj": True,
  "task_specific_params": {
    "text-generation": {
      "do_sample": True,
      "max_length": 50
    }
  },
  "vocab_size": 50257
}
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
gpt = TransformerDecoder(
    num_layers=GPT2Config['n_layer'],
    num_heads=GPT2Config['n_head'],
    vocab_size=GPT2Config['vocab_size'],
    context_len=GPT2Config['n_ctx'],
    embedding_dim=GPT2Config['n_embd'],
)

In [5]:
print(*(x+'\n' for x in gpt.state_dict().keys()))

token_embedding.weight
 positional_embedding.weight
 ln_f.weight
 ln_f.bias
 blocks.0.mha.attn_heads.0.Q_proj.weight
 blocks.0.mha.attn_heads.0.K_proj.weight
 blocks.0.mha.attn_heads.0.V_proj.weight
 blocks.0.mha.attn_heads.1.Q_proj.weight
 blocks.0.mha.attn_heads.1.K_proj.weight
 blocks.0.mha.attn_heads.1.V_proj.weight
 blocks.0.mha.attn_heads.2.Q_proj.weight
 blocks.0.mha.attn_heads.2.K_proj.weight
 blocks.0.mha.attn_heads.2.V_proj.weight
 blocks.0.mha.attn_heads.3.Q_proj.weight
 blocks.0.mha.attn_heads.3.K_proj.weight
 blocks.0.mha.attn_heads.3.V_proj.weight
 blocks.0.mha.attn_heads.4.Q_proj.weight
 blocks.0.mha.attn_heads.4.K_proj.weight
 blocks.0.mha.attn_heads.4.V_proj.weight
 blocks.0.mha.attn_heads.5.Q_proj.weight
 blocks.0.mha.attn_heads.5.K_proj.weight
 blocks.0.mha.attn_heads.5.V_proj.weight
 blocks.0.mha.attn_heads.6.Q_proj.weight
 blocks.0.mha.attn_heads.6.K_proj.weight
 blocks.0.mha.attn_heads.6.V_proj.weight
 blocks.0.mha.attn_heads.7.Q_proj.weight
 blocks.0.mha.attn_hea

------------------

In [6]:
import os
path = os.getcwd()
state_dict = torch.load(path+'/weights/pytorch_model.bin')

In [7]:
state_dict['h.0.attn.c_attn.bias']

tensor([ 0.4803, -0.5254, -0.4293,  ...,  0.0126, -0.0499,  0.0032])

In [8]:
import torch.nn as nn
nn.Linear?

Init signature:
nn.Linear(
    in_features: int,
    out_features: int,
    bias: bool = True,
    device=None,
    dtype=None,
) -> None
Docstring:     
Applies an affine linear transformation to the incoming data: :math:`y = xA^T + b`.

This module supports :ref:`TensorFloat32<tf32_on_ampere>`.

On certain ROCm devices, when using float16 inputs this module will use :ref:`different precision<fp16_on_mi200>` for backward.

Args:
    in_features: size of each input sample
    out_features: size of each output sample
    bias: If set to ``False``, the layer will not learn an additive bias.
        Default: ``True``

Shape:
    - Input: :math:`(*, H_{in})` where :math:`*` means any number of
      dimensions including none and :math:`H_{in} = \text{in\_features}`.
    - Output: :math:`(*, H_{out})` where all but the last dimension
      are the same shape as the input and :math:`H_{out} = \text{out\_features}`.

Attributes:
    weight: the learnable weights of the module of shape
       

In [9]:
"""
wte is probably token embedding
wpe is positional embedding
lnf ? probably last linear layer (no 50257 because of tied (shared) embedding weights?)
h.0-11 are the heads
     - ln layer norm
     - mlp is ffn
     - attn.bias vs attn.c_attn ??
     - c_attn is the Q, K, V projection of all 12 heads at once
     - c_proj is the out_proj, the final part of MHA, NOT the MLP in the block
this is done using Conv1D (from OpenAI) which is a linear layer (normally xA^T + b, TxN @ FxN^T + TxF) 
without the weight matrix transpose (so Ax + b, FxN @ TxN^T + FxT) ?
anyways, we prolly need to transpose this matrix from the state dict to fit into normal linear layers (also for mlp)
"""
for i in state_dict.keys():
    print(i, '==>', list(state_dict[i].shape))

wte.weight ==> [50257, 768]
wpe.weight ==> [1024, 768]
h.0.ln_1.weight ==> [768]
h.0.ln_1.bias ==> [768]
h.0.attn.bias ==> [1, 1, 1024, 1024]
h.0.attn.c_attn.weight ==> [768, 2304]
h.0.attn.c_attn.bias ==> [2304]
h.0.attn.c_proj.weight ==> [768, 768]
h.0.attn.c_proj.bias ==> [768]
h.0.ln_2.weight ==> [768]
h.0.ln_2.bias ==> [768]
h.0.mlp.c_fc.weight ==> [768, 3072]
h.0.mlp.c_fc.bias ==> [3072]
h.0.mlp.c_proj.weight ==> [3072, 768]
h.0.mlp.c_proj.bias ==> [768]
h.1.ln_1.weight ==> [768]
h.1.ln_1.bias ==> [768]
h.1.attn.bias ==> [1, 1, 1024, 1024]
h.1.attn.c_attn.weight ==> [768, 2304]
h.1.attn.c_attn.bias ==> [2304]
h.1.attn.c_proj.weight ==> [768, 768]
h.1.attn.c_proj.bias ==> [768]
h.1.ln_2.weight ==> [768]
h.1.ln_2.bias ==> [768]
h.1.mlp.c_fc.weight ==> [768, 3072]
h.1.mlp.c_fc.bias ==> [3072]
h.1.mlp.c_proj.weight ==> [3072, 768]
h.1.mlp.c_proj.bias ==> [768]
h.2.ln_1.weight ==> [768]
h.2.ln_1.bias ==> [768]
h.2.attn.bias ==> [1, 1, 1024, 1024]
h.2.attn.c_attn.weight ==> [768, 2304]

In [10]:
new_state_dict = {}

# TODO:
# - split pretrained attention matrix into distinct 12 heads for my implementation
#   and add biases for attn_heads, then do the same for them. (i thought gpt2 had bias=False?)
# - put tied weights into each component (which one? last linear layer? which layer does the "out" embedding?)
for my_key, pretrained_key in zip(gpt.state_dict().keys(), state_dict.keys()):
    print(my_key, pretrained_key)
    print(list(gpt.state_dict()[my_key].shape), list(state_dict[pretrained_key].shape))
    new_state_dict[my_key] = state_dict[pretrained_key]

token_embedding.weight wte.weight
[50257, 768] [50257, 768]
positional_embedding.weight wpe.weight
[1024, 768] [1024, 768]
ln_f.weight h.0.ln_1.weight
[768] [768]
ln_f.bias h.0.ln_1.bias
[768] [768]
blocks.0.mha.attn_heads.0.Q_proj.weight h.0.attn.bias
[64, 768] [1, 1, 1024, 1024]
blocks.0.mha.attn_heads.0.K_proj.weight h.0.attn.c_attn.weight
[64, 768] [768, 2304]
blocks.0.mha.attn_heads.0.V_proj.weight h.0.attn.c_attn.bias
[64, 768] [2304]
blocks.0.mha.attn_heads.1.Q_proj.weight h.0.attn.c_proj.weight
[64, 768] [768, 768]
blocks.0.mha.attn_heads.1.K_proj.weight h.0.attn.c_proj.bias
[64, 768] [768]
blocks.0.mha.attn_heads.1.V_proj.weight h.0.ln_2.weight
[64, 768] [768]
blocks.0.mha.attn_heads.2.Q_proj.weight h.0.ln_2.bias
[64, 768] [768]
blocks.0.mha.attn_heads.2.K_proj.weight h.0.mlp.c_fc.weight
[64, 768] [768, 3072]
blocks.0.mha.attn_heads.2.V_proj.weight h.0.mlp.c_fc.bias
[64, 768] [3072]
blocks.0.mha.attn_heads.3.Q_proj.weight h.0.mlp.c_proj.weight
[64, 768] [3072, 768]
blocks.0.mh

[64, 768] [768]
blocks.3.mha.attn_heads.6.K_proj.weight ln_f.bias
[64, 768] [768]


In [11]:
print(*(x+'\n' for x in gpt.state_dict().keys()))

token_embedding.weight
 positional_embedding.weight
 ln_f.weight
 ln_f.bias
 blocks.0.mha.attn_heads.0.Q_proj.weight
 blocks.0.mha.attn_heads.0.K_proj.weight
 blocks.0.mha.attn_heads.0.V_proj.weight
 blocks.0.mha.attn_heads.1.Q_proj.weight
 blocks.0.mha.attn_heads.1.K_proj.weight
 blocks.0.mha.attn_heads.1.V_proj.weight
 blocks.0.mha.attn_heads.2.Q_proj.weight
 blocks.0.mha.attn_heads.2.K_proj.weight
 blocks.0.mha.attn_heads.2.V_proj.weight
 blocks.0.mha.attn_heads.3.Q_proj.weight
 blocks.0.mha.attn_heads.3.K_proj.weight
 blocks.0.mha.attn_heads.3.V_proj.weight
 blocks.0.mha.attn_heads.4.Q_proj.weight
 blocks.0.mha.attn_heads.4.K_proj.weight
 blocks.0.mha.attn_heads.4.V_proj.weight
 blocks.0.mha.attn_heads.5.Q_proj.weight
 blocks.0.mha.attn_heads.5.K_proj.weight
 blocks.0.mha.attn_heads.5.V_proj.weight
 blocks.0.mha.attn_heads.6.Q_proj.weight
 blocks.0.mha.attn_heads.6.K_proj.weight
 blocks.0.mha.attn_heads.6.V_proj.weight
 blocks.0.mha.attn_heads.7.Q_proj.weight
 blocks.0.mha.attn_hea

In [12]:
mygptdict = gpt.state_dict()

mygptdict['token_embedding.weight'].copy_(state_dict['wte.weight'])
mygptdict['positional_embedding.weight'].copy_(state_dict['wpe.weight'])

tensor([[-1.8821e-02, -1.9742e-01,  4.0267e-03,  ..., -4.3044e-02,
          2.8267e-02,  5.4490e-02],
        [ 2.3959e-02, -5.3792e-02, -9.4879e-02,  ...,  3.4170e-02,
          1.0172e-02, -1.5573e-04],
        [ 4.2161e-03, -8.4764e-02,  5.4515e-02,  ...,  1.9745e-02,
          1.9325e-02, -2.1424e-02],
        ...,
        [-1.7987e-03,  1.6052e-03, -5.5103e-02,  ...,  1.3617e-02,
         -7.1805e-03,  3.7552e-03],
        [ 3.2105e-03,  1.5501e-03, -4.8944e-02,  ...,  2.0725e-02,
         -1.1838e-02, -5.5683e-04],
        [ 2.6610e-04,  3.0272e-03, -1.7086e-03,  ..., -4.6506e-03,
         -2.3541e-03, -5.7855e-03]])

In [13]:
gpt.state_dict()['token_embedding.weight'],gpt.state_dict()['positional_embedding.weight']

(tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
         [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
         [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
         ...,
         [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
         [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
         [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]]),
 tensor([[-1.8821e-02, -1.9742e-01,  4.0267e-03,  ..., -4.3044e-02,
           2.8267e-02,  5.4490e-02],
         [ 2.3959e-02, -5.3792e-02, -9.4879e-02,  ...,  3.4170e-02,
           1.0172e-02, -1.5573e-04],
         [ 4.2161e-03, -8.4764e-02,  5.4515e-02,  ...,  1.9745e-02,
           1.9325e-02, -2.1424e-02],
         ...,
         [-1.7987e-03,  1.6052e-03, -5.5103e-02,  ...,  1.3617e-02,
          -7.1805e-03,  3.7552e-03],
         [ 3.2105e-03,  1.5501e-03, -4.8944e-02,  ...,  2.0725e-02,
          -1.1838e-02, -5.5683e-04],
         [ 2.6610e-

In [14]:
state_dict['wte.weight'], state_dict['wpe.weight']

(tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
         [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
         [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
         ...,
         [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
         [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
         [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]]),
 tensor([[-1.8821e-02, -1.9742e-01,  4.0267e-03,  ..., -4.3044e-02,
           2.8267e-02,  5.4490e-02],
         [ 2.3959e-02, -5.3792e-02, -9.4879e-02,  ...,  3.4170e-02,
           1.0172e-02, -1.5573e-04],
         [ 4.2161e-03, -8.4764e-02,  5.4515e-02,  ...,  1.9745e-02,
           1.9325e-02, -2.1424e-02],
         ...,
         [-1.7987e-03,  1.6052e-03, -5.5103e-02,  ...,  1.3617e-02,
          -7.1805e-03,  3.7552e-03],
         [ 3.2105e-03,  1.5501e-03, -4.8944e-02,  ...,  2.0725e-02,
          -1.1838e-02, -5.5683e-04],
         [ 2.6610e-

In [15]:
for i in range(12):
    print(state_dict[f'h.{i}.ln_1.weight'])
    print(state_dict[f'h.{i}.ln_1.bias'])
    print(state_dict[f'h.{i}.attn.bias'])
    print(state_dict[f'h.{i}.attn.c.weight'])
    print(state_dict[f'h.{i}.ln_1.weight'])

    """
h.0.ln_1.weight ==> [768]
h.0.ln_1.bias ==> [768]
h.0.attn.bias ==> [1, 1, 1024, 1024]
h.0.attn.c_attn.weight ==> [768, 2304]
h.0.attn.c_attn.bias ==> [2304]
h.0.attn.c_proj.weight ==> [768, 768]
h.0.attn.c_proj.bias ==> [768]
h.0.ln_2.weight ==> [768]
h.0.ln_2.bias ==> [768]
h.0.mlp.c_fc.weight ==> [768, 3072]
h.0.mlp.c_fc.bias ==> [3072]
h.0.mlp.c_proj.weight ==> [3072, 768]
h.0.mlp.c_proj.bias ==> [768]

    """

tensor([0.2232, 0.1820, 0.1534, 0.1917, 0.2036, 0.1948, 0.1467, 0.1865, 0.2143,
        0.1956, 0.2118, 0.2153, 0.1882, 0.2074, 0.1871, 0.2040, 0.2044, 0.1900,
        0.1952, 0.0475, 0.1909, 0.2115, 0.1971, 0.2202, 0.1998, 0.2108, 0.2303,
        0.1879, 0.1939, 0.2018, 0.1891, 0.1861, 0.1958, 0.1832, 0.1978, 0.2243,
        0.0706, 0.1958, 0.1943, 0.1939, 0.1978, 0.1951, 0.1995, 0.1912, 0.2083,
        0.2037, 0.1849, 0.1945, 0.2189, 0.0419, 0.1977, 0.1979, 0.0608, 0.1824,
        0.2055, 0.0476, 0.1892, 0.2079, 0.2047, 0.2233, 0.2097, 0.2075, 0.2076,
        0.1793, 0.1312, 0.1841, 0.1939, 0.1561, 0.0577, 0.1948, 0.2048, 0.1717,
        0.1942, 0.1708, 0.1989, 0.1993, 0.2082, 0.1071, 0.1968, 0.1770, 0.2164,
        0.1864, 0.1938, 0.2184, 0.1343, 0.1707, 0.0683, 0.1401, 0.1823, 0.2045,
        0.2007, 0.1853, 0.1783, 0.1889, 0.1870, 0.1975, 0.2114, 0.2108, 0.2083,
        0.2409, 0.1938, 0.2022, 0.0857, 0.1823, 0.1879, 0.1979, 0.1850, 0.1029,
        0.1762, 0.1953, 0.2231, 0.2006, 

KeyError: 'h.0.attn.c.weight'

In [16]:
print(state_dict['h.0.attn.c_attn.weight'].shape)

torch.Size([768, 2304])


{
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_layer": 12,
  "n_positions": 1024
}

In [18]:
max_positions = 1024
torch.tril(torch.ones((max_positions, max_positions), dtype=torch.bool)).view(
                1, 1, max_positions, max_positions)

tensor([[[[ True, False, False,  ..., False, False, False],
          [ True,  True, False,  ..., False, False, False],
          [ True,  True,  True,  ..., False, False, False],
          ...,
          [ True,  True,  True,  ...,  True, False, False],
          [ True,  True,  True,  ...,  True,  True, False],
          [ True,  True,  True,  ...,  True,  True,  True]]]])

In [87]:
torch.tril(torch.ones((1024,1024), dtype=torch.bool)).view(1,1,1024,1024)

tensor([[[[ True, False, False,  ..., False, False, False],
          [ True,  True, False,  ..., False, False, False],
          [ True,  True,  True,  ..., False, False, False],
          ...,
          [ True,  True,  True,  ...,  True, False, False],
          [ True,  True,  True,  ...,  True,  True, False],
          [ True,  True,  True,  ...,  True,  True,  True]]]])

This is the causal part in the attention. We're attending only to past tokens and the current one by using this mask on the context window (1024x784)(?).

I guess the 1's added in the .view() are for batch size and(?) `n_tokens`? is the batchsize equal to the number of input tokens? like, is that the same thing?

In [90]:
causal_mask = torch.tril(torch.ones((1024,1024), dtype=torch.bool)).view(1,1,1024,1024)
torch.where(?)

In [128]:
torch.finfo(torch.float16)

finfo(resolution=0.001, min=-65504, max=65504, eps=0.000976562, smallest_normal=6.10352e-05, tiny=6.10352e-05, dtype=float16)

In [129]:
torch.finfo(torch.float32)

finfo(resolution=1e-06, min=-3.40282e+38, max=3.40282e+38, eps=1.19209e-07, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=float32)

In [125]:
torch.tensor(1, dtype=torch.float16)+torch.tensor(0.0005, dtype=torch.float16)

tensor(1.0010, dtype=torch.float16)

In [5]:
import os
path = os.getcwd()
state_dict = torch.load(path+'/weights/pytorch_model.bin')

In [8]:
state_dict['h.11.attn.bias']

tensor([[[[1., 0., 0.,  ..., 0., 0., 0.],
          [1., 1., 0.,  ..., 0., 0., 0.],
          [1., 1., 1.,  ..., 0., 0., 0.],
          ...,
          [1., 1., 1.,  ..., 1., 0., 0.],
          [1., 1., 1.,  ..., 1., 1., 0.],
          [1., 1., 1.,  ..., 1., 1., 1.]]]])

In [13]:
torch.where(torch.arange(1024*1024).view(1024, 1024))

tensor([[      0,       1,       2,  ...,    1021,    1022,    1023],
        [   1024,    1025,    1026,  ...,    2045,    2046,    2047],
        [   2048,    2049,    2050,  ...,    3069,    3070,    3071],
        ...,
        [1045504, 1045505, 1045506,  ..., 1046525, 1046526, 1046527],
        [1046528, 1046529, 1046530,  ..., 1047549, 1047550, 1047551],
        [1047552, 1047553, 1047554,  ..., 1048573, 1048574, 1048575]])

In [ ]:
causal_mask = state_dict['h.11.attn.bias'][:, :, 768 - query_length : key_length, :key_length]

In [ ]:
self.mask.bool()[:n_tokens, :n_tokens], -torch.inf) 


In [26]:
causal_mask = torch.tril(torch.ones((1024,1024), dtype=torch.bool)).view(1, 1, 1024,1024)


In [27]:
causal_mask

tensor([[[[ True, False, False,  ..., False, False, False],
          [ True,  True, False,  ..., False, False, False],
          [ True,  True,  True,  ..., False, False, False],
          ...,
          [ True,  True,  True,  ...,  True, False, False],
          [ True,  True,  True,  ...,  True,  True, False],
          [ True,  True,  True,  ...,  True,  True,  True]]]])

In [36]:
attn_weights = torch.randn(3,3)
mask_val = torch.finfo(attn_weights.dtype).min
causal_mask[:, :, :3, :3], attn_weights, mask_val

(tensor([[[[ True, False, False],
           [ True,  True, False],
           [ True,  True,  True]]]]),
 tensor([[ 0.0578, -1.2552,  0.8472],
         [-1.5971,  1.0757,  1.6850],
         [-0.7978,  0.1029,  0.4572]]),
 -3.4028234663852886e+38)

In [52]:
from torch.nn.functional import softmax

In [49]:
sdp = torch.where(causal_mask[:, :, :3, :3], attn_weights, 0)
print(sdp)
print(softmax(sdp, dim=-1))

tensor([[[[ 0.0578,  0.0000,  0.0000],
          [-1.5971,  1.0757,  0.0000],
          [-0.7978,  0.1029,  0.4572]]]])
tensor([[[[0.3463, 0.3269, 0.3269],
          [0.0490, 0.7092, 0.2419],
          [0.1435, 0.3532, 0.5033]]]])


In [50]:
sdp = torch.where(causal_mask[:, :, :3, :3], attn_weights, 1e-9)
print(sdp)
print(softmax(sdp, dim=-1))

tensor([[[[ 5.7769e-02,  1.0000e-09,  1.0000e-09],
          [-1.5971e+00,  1.0757e+00,  1.0000e-09],
          [-7.9779e-01,  1.0285e-01,  4.5718e-01]]]])
tensor([[[[0.3463, 0.3269, 0.3269],
          [0.0490, 0.7092, 0.2419],
          [0.1435, 0.3532, 0.5033]]]])


In [51]:
sdp = torch.where(causal_mask[:, :, :3, :3], attn_weights, mask_val)
print(sdp)
print(softmax(sdp, dim=-1))

tensor([[[[ 5.7769e-02, -3.4028e+38, -3.4028e+38],
          [-1.5971e+00,  1.0757e+00, -3.4028e+38],
          [-7.9779e-01,  1.0285e-01,  4.5718e-01]]]])
tensor([[[[1.0000, 0.0000, 0.0000],
          [0.0646, 0.9354, 0.0000],
          [0.1435, 0.3532, 0.5033]]]])
